# 2.2_huim — Candidate-aware attention for relevance shortlisting

This notebook runs one final workflow: **Optuna → apply the winning parameters → train on all recorded training requests → restore the selected validation checkpoint → export all requests → save and reload from shared Drive**.

Stage 1 scores every time-eligible article in the shared catalogue and retains its genuine **top 200**. Stage 2 applies the business priorities and displays five recommendations. Selection remains **known-click Recall@100**, then observed-click nDCG@100, then the earlier checkpoint/trial.

Earlier bounded checks do not establish this final run's results. The cells are cleared for the final Colab execution; report only outcomes produced by that execution.


## 1. Business hypothesis and experiment overview

| Item | Definition |
| --- | --- |
| Experiment / owner | `2.2_huim` / Winston Ho Rui Wei |
| Business objective | Relevant recommendations that support continued reading and reader retention (Interim Report §1.1, §1.2.3) |
| Hypothesis | Readers have several interests. Matching each candidate to individual past clicks should recover more of their known interests than one history average. |
| Model | HUIM-inspired candidate-conditioned attention over fixed history events, article metadata and frozen text/image embeddings; a separate request/pair feature branch and MLP produce click logits. |
| Evidence | Development validation Recall@100 and observed-click nDCG@100 on the declared eligible pool; recorded-slate AUC is a diagnostic. Later shared evaluation will examine representative recommendations and compare models on identical requests. |

The article tower projects frozen 768-dimensional text and 1,024-dimensional image vectors and combines them with metadata embeddings and missingness flags. Each history event also carries train-fitted reading-depth bins and a recency bin. Attention uses `[history, candidate, difference, product]`, a small MLP and masked softmax. The model has no recurrence or positional encoding: recency belongs to each event, and permuting aligned events does not change a prediction. The final feature fusion uses concatenation and LayerNorm.

This adapts HUIM's target attention and feature interactions. It does not reproduce its whole-catalogue PCA, slate-dependent popularity features, history augmentation or reported paper scores. Fixed histories contain events strictly before the split's first request. Training uses BCE on the supplied recorded candidates: shown-clicked = 1, shown-unclicked = 0. Unshown catalogue articles have unknown outcomes and never become training negatives.

**Selection:** macro known-click Recall@100 over the same 100 validation requests, each scored against the full eligible catalogue; observed-click nDCG@100 breaks ties and exact ties retain the earlier checkpoint/trial. All original clicks remain in recall denominators and ideal DCG, including clicks rejected by eligibility. These are development results because validation is reused for selection. Test has no labels and is never used for tuning.

Course connection: the implementation combines categorical embeddings, attention, feature interactions, dense networks and dropout/weight-decay regularisation. The complete HUIM-inspired architecture is an adaptation rather than a model taught as a whole in class.


## 2. Setup and run configuration

Use a Colab GPU runtime. Upload `huim_helpers.zip` beside this notebook to `/content`, or place the editable `huim/` folder there. Locally, the same ZIP or folder can sit beside the notebook. The ZIP distributed with this notebook contains exactly the editable helper sources.

Add the shared folder shortcut under `MyDrive/Colab Notebooks/BT4222 Shared Final Folder`. Inputs stay there; the complete run is saved under `Experiments/2.2_huim/<run_id>/` in that folder.


In [ ]:
from pathlib import Path
import time
from datetime import datetime
from zoneinfo import ZoneInfo

SEED = 42
OWNER = "Winston Ho Rui Wei"
DATA_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder")
OUTPUT_ROOT = DATA_ROOT / "Experiments" / "2.2_huim"
LOCAL_ROOT = Path("/content/huim_runs")
RUN_ID = f"{datetime.now(ZoneInfo('Asia/Singapore')):%Y%m%d_%H%M%S}_seed{SEED}"
TUNING_TRIALS = 12
TUNING_MAX_EPOCHS = 2
TUNING_TRAIN_REQUESTS = 60_000


In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Local execution: set DATA_ROOT and LOCAL_ROOT above to your local folders.")
assert DATA_ROOT.is_dir(), f"Shared data folder not found: {DATA_ROOT}"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)

import importlib, subprocess, sys
for module, package in {"numpy": "numpy>=1.24", "pandas": "pandas>=2.0", "pyarrow": "pyarrow>=14",
                        "torch": "torch>=2.3", "optuna": "optuna>=3.6,<5", "duckdb": "duckdb>=1.0"}.items():
    if importlib.util.find_spec(module) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)


In [ ]:
import zipfile, hashlib, json, shutil
roots = [Path("/content"), Path.cwd()]
EXP_DIR = next((root for root in roots if (root / "huim_helpers.zip").is_file()
                or (root / "huim" / "__init__.py").is_file()), None)
assert EXP_DIR is not None, "Upload huim_helpers.zip or the huim/ folder beside the notebook."
if (EXP_DIR / "huim_helpers.zip").is_file():
    with zipfile.ZipFile(EXP_DIR / "huim_helpers.zip") as archive:
        archive.extractall(EXP_DIR)
for name in [name for name in sys.modules if name == "huim" or name.startswith("huim.")]:
    del sys.modules[name]
sys.path.insert(0, str(EXP_DIR))
import huim
from huim import pipeline as P, catalogue as C, features as F, data as D
from huim.io_utils import RunLogger, atomic_write_json, environment_info, read_json
from huim.model import parameter_counts
import numpy as np, pandas as pd, torch, optuna

HELPER_HASHES = {path.name: hashlib.sha256(path.read_bytes()).hexdigest()
                 for path in sorted((EXP_DIR / "huim").glob("*.py"))}
CODE_SOURCE = {"source": "uploaded huim_helpers.zip or adjacent huim folder", "commit": None,
               "uncommitted_changes": None, "helper_sha256": HELPER_HASHES,
               "note": "Exact executed helper code is copied to model/code/huim; no Git revision is inferred."}
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CFG = P.make_config(seed=SEED, max_epochs=4, train_requests=None, selection_requests=100,
                    export_requests={split: None for split in D.SPLITS}, selection_interval_steps=300)
RUN_DIR, PERSIST_DIR = LOCAL_ROOT / RUN_ID, OUTPUT_ROOT / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=False)
log = RunLogger(RUN_DIR)
log(f"run {RUN_ID}; device {DEVICE}; all training/export requests; selection=100; export K=200")
print("Training AMP:", CFG["amp"] and DEVICE.type == "cuda")
print("Tuning:", TUNING_TRIALS, "trials ×", TUNING_MAX_EPOCHS, "epochs, up to", TUNING_TRAIN_REQUESTS, "complete requests/trial")


## 3. Load and check shared data

Load the supplied recorded pair keys/labels, behaviours and fixed histories. Build **one ordered article union: train_val, then test; first source record wins, including missing fields**. The same union and vector precedence apply to every split. An article is eligible only when its known publication time is at or before the request time. Missing embeddings keep the article eligible under the model's documented missing-vector fallback.

All training and export requests are retained. The fixed 100 validation-selection requests use the shared seed-42 BLAKE2b request rule used by 3.1. Each selected request uses its complete eligible article pool. Input preparation decodes each vector source once and records reader, dimensions, missingness and source precedence.


In [ ]:
with log.stage("prepare shared inputs"):
    ws = P.prepare_inputs(CFG, DATA_ROOT, log=log)
    INPUT_MANIFEST = D.input_manifest(ws.paths)
    SCOPES = P.resolve_request_scopes(ws, CFG)
    atomic_write_json(RUN_DIR / "input_manifest.json", INPUT_MANIFEST)
    atomic_write_json(RUN_DIR / "selection" / "request_scopes.json",
                      {"pool_fingerprint": ws.reports["pool_fingerprint"], "pool_policy": ws.reports["pool_policy"],
                       **P.scope_ids(ws, SCOPES)})
    PREFLIGHT = P.preflight(ws, CFG, [LOCAL_ROOT, OUTPUT_ROOT], SCOPES)
    atomic_write_json(RUN_DIR / "preflight.json", PREFLIGHT)
BUDGET = PREFLIGHT["budget"]
display(pd.DataFrame({split: {"requests": len(ws.ctx[split].requests),
                              "recorded_pairs": len(ws.pairs[split]),
                              "catalogue_articles": ws.ctx[split].report["catalogue_articles"],
                              "eligible_pairs_to_score": BUDGET[split]["planned_export"]["eligible_pairs"],
                              "retained_score_rows": BUDGET[split]["planned_export"]["retained_score_rows"],
                              "retained_output_GB_estimate": BUDGET[split]["planned_export"]["retained_output_bytes_estimate"] / 1e9}
                      for split in D.SPLITS}).T)
print("Pool:", ws.reports["pool_fingerprint"])
print("Embedding coverage:", PREFLIGHT["coverage"])
print("Recorded candidates rejected by catalogue eligibility:", PREFLIGHT["recorded_ineligible"])
print("Selection pairs per evaluation:", BUDGET["selection_per_evaluation_pairs"])
print("Output estimates use uncompressed retained row/list sizes; actual Parquet sizes depend on compression.")


## 4. Prepare model inputs

Request/article/history features are constructed for arbitrary pairs with one shared inference path. Vocabularies, engagement bins, body scaling and dense standardisation are fitted on permitted training data. Publication cutoff, category preferences and missingness indicators use only request-time context; slate size, in-slate rank, current reading outcomes and later behaviour clicks are excluded.

The saved feature dictionary preserves ordered feature names and category index-to-value maps. `preprocessing.json` preserves the fitted transforms and `article_ids.npy` the original article-ID order. Embedding coordinates are representation dimensions, not word-level explanations.


In [ ]:
with log.stage("fit training preprocessing"):
    F.fit_dense_scalers(ws.pre, ws.ctx["train"], ws.art)
FEATURE_ORDER = {"request_context": F.REQUEST_DENSE_COLUMNS(ws.pre), "candidate": F.PAIR_DENSE_COLUMNS}
display(pd.DataFrame(F.FEATURE_AUDIT, columns=["feature", "scope", "status", "reason"]))
print("Named dense inputs:", FEATURE_ORDER)


## 5. Define and train model

**Mandatory bounded search:** 12 seeded TPE trials, at most two epochs and 60,000 complete training requests per trial. Search learning rate, weight decay, dropout, article dimension and modality dropout. Every trial uses the same model-training seed (42), the same training request prefix and the fixed validation-selection requests. Median pruning uses intermediate catalogue Recall@100. Save every trial outcome and a consistent SQLite study snapshot to shared Drive.

Apply the winner before the main fit on **all recorded training requests**. AdamW, GPU mixed precision and gradient clipping optimise BCE. Evaluate Recall@100 every 300 steps and at each epoch end; the scheduler and early stopping follow that objective. The main budget is four epochs, subject to early stopping. Restore and re-evaluate the selected checkpoint before exporting.


In [ ]:
from huim.tune import run_study, select_best_trial, SEARCH_SPACE
SELECTOR = P.make_selector(ws, CFG, DEVICE, SCOPES)
def objective(trial, params):
    trial_cfg = dict(CFG, **params, max_epochs=TUNING_MAX_EPOCHS, train_requests=TUNING_TRAIN_REQUESTS,
                     checkpoint_every_steps=0)
    def report(record):
        trial.report(record["known_click_recall@100"], record["eval_index"])
        if trial.should_prune():
            raise optuna.TrialPruned()
    _, _, state, subset, _ = P.train_model(
        ws, trial_cfg, RUN_DIR / "tuning" / f"trial_{trial.number:03d}", None, DEVICE,
        resume=False, log=log, selector=SELECTOR, on_evaluation=report)
    best = state["best"]
    for name in ("observed_click_ndcg@100", "diagnostic_recorded_slate_auc", "epoch", "global_step"):
        trial.set_user_attr(name, best[name])
    trial.set_user_attr("training_scope", subset)
    return best["known_click_recall@100"]

with log.stage("Optuna validation search"):
    study = run_study(objective, f"huim_{RUN_ID}", RUN_DIR / "tuning", PERSIST_DIR, TUNING_TRIALS, SEED)
winner = select_best_trial(study)
assert winner is not None, "No trial completed; inspect saved tuning/trials.csv."
CFG.update(winner["params"])
STUDY_SUMMARY = {"search_space": SEARCH_SPACE, "actual_trials": len(study.trials),
                 "states": pd.Series([trial.state.name for trial in study.trials]).value_counts().to_dict(),
                 "selected_trial": winner, "applied_to_main_training": True,
                 "sampler": "TPESampler(seed=42)",
                 "pruner": "MedianPruner(n_startup_trials=3, n_warmup_steps=1)",
                 "objective": "Recall@100, then observed-click nDCG@100, then earlier trial",
                 "model_seed_every_trial": SEED, "selection_sample": SELECTOR.protocol["sample"],
                 "budget": {"trials": TUNING_TRIALS, "max_epochs": TUNING_MAX_EPOCHS,
                            "training_requests_per_trial": TUNING_TRAIN_REQUESTS}}
atomic_write_json(RUN_DIR / "tuning" / "summary.json", STUDY_SUMMARY)
atomic_write_json(RUN_DIR / "tuning" / "tuned_config.json", CFG)
for name in ("summary.json", "tuned_config.json"):
    shutil.copyfile(RUN_DIR / "tuning" / name, PERSIST_DIR / "tuning" / name)
display(study.trials_dataframe())
print("Applied winning parameters:", winner["params"])


In [ ]:
with log.stage("fit all recorded training requests"):
    model, MODEL_CFG, TRAIN_STATE, TRAIN_SUBSET, SELECTION = P.train_model(
        ws, CFG, RUN_DIR, PERSIST_DIR, DEVICE, resume=False, log=log, selector=SELECTOR)
assert TRAIN_SUBSET["requests"] == len(ws.ctx["train"].requests)
assert SELECTION["restored_matches_recorded"], "Selected checkpoint re-evaluation disagrees with its recorded result."
# Preserve the selected model and every fitted transform before catalogue export.
shutil.copytree(RUN_DIR / "model", PERSIST_DIR / "model", dirs_exist_ok=True)
shutil.copytree(RUN_DIR / "selection", PERSIST_DIR / "selection", dirs_exist_ok=True)
shutil.copyfile(RUN_DIR / "training_history.json", PERSIST_DIR / "training_history.json")
PARAMS = parameter_counts(model)
print("Actual stop reason:", TRAIN_STATE["stop_reason"])
print("Selected checkpoint:", SELECTION["selected_checkpoint"])
print("Parameters:", PARAMS)


## 6. Validation results

Report the selected checkpoint on the fixed 100-request selection sample and recorded-slate AUC separately. All-validation Recall@100 and Recall@200 follow from the export below. Saved training histories and Optuna trials support later learning curves and search analysis. Detailed attention examples, robustness checks, feature analysis and cross-model plots belong in the planned shared evaluation notebook.


In [ ]:
with log.stage("selected validation results"):
    SELECTED_SAMPLE = SELECTOR.evaluate(model)
    art_d, contexts_d = P.device_and_tensors(ws, DEVICE)
    VAL_DIAGNOSTIC, _ = P.evaluate_recorded(model, contexts_d["validation"], art_d, DEVICE)
display(pd.Series({key: SELECTED_SAMPLE[key] for key in (
    "known_click_recall@100", "observed_click_ndcg@100", "eligible_only_recall@100",
    "requests_evaluated", "clicked_articles_total", "clicked_articles_eligible")}))
print("Recorded-slate diagnostic:", VAL_DIAGNOSTIC)


## 7. Predict and export all splits

For every train, validation and test request, score the **complete eligible union catalogue** in bounded chunks. Keep the genuine top `min(200, eligible_count)`, ordered by descending logit then ascending original article ID. A running top-K retains only 201 items per request while checking each merge, including ties at 100 and 200. Never insert recorded candidates or known clicks. Empty pools remain explicit as `[]`.

| Artifact | Contents |
| --- | --- |
| `predictions/{split}/submission_article_scores.parquet` | Retained top-200 pairs: string behaviour ID, int64 article ID, 1-based rank, float32 logit and its uncalibrated sigmoid probability |
| `rankings/{split}/submission_top_200.parquet` | Every request, native list<int64> in the same order |
| `diagnostics/recorded_slates/{split}_scores.parquet` | Every original shown candidate separately, including shown nonclicks and articles outside the shortlist; labels are joined separately |
| `diagnostics/catalogue/{split}_requests.parquet` | IDs, timestamps, eligible counts, empty pools and known-click retrieval bookkeeping |

No exhaustive catalogue matrix or score CSV is exported. Selection K stays 100; export K and shortlist size are 200; final display size is five. Unknown/unshown outcomes remain unknown, and no test labels or metrics are invented.


In [ ]:
with log.stage("all recorded-slate diagnostic scores"):
    RECORDED, recorded_logits = P.export_recorded(model, ws, RUN_DIR, DEVICE)
    del recorded_logits
assert all(result["validation"]["passed"] for result in RECORDED.values())
with log.stage("all-request catalogue top-200 exports"):
    EXPORTS = P.export_catalogue(model, ws, CFG, RUN_DIR, DEVICE,
                                  {split: SCOPES[f"export_{split}"]["positions"] for split in D.SPLITS}, log=log)
for split in D.SPLITS:
    assert EXPORTS[split]["summary"]["requests"] == len(ws.ctx[split].requests)
EXPORTED_VALIDATION = EXPORTS["validation"]["summary"]["recall"]
display(pd.DataFrame({split: result["summary"] | {"validated": result["validation"]["passed"]}
                      for split, result in EXPORTS.items()}).T)
print("All-validation development retrieval metrics:", EXPORTED_VALIDATION)


In [ ]:
contract = {split: {
    "scores": f"predictions/{split}/submission_article_scores.parquet",
    "rankings": f"rankings/{split}/submission_top_200.parquet",
    "recorded_slates": f"diagnostics/recorded_slates/{split}_scores.parquet",
    "request_manifest": f"diagnostics/catalogue/{split}_requests.parquet",
    "request_scope": SCOPES[f"export_{split}"]["info"], **EXPORTS[split]["summary"],
    "validation": EXPORTS[split]["validation"],
} for split in D.SPLITS}
common = {"export_schema_version": C.EXPORT_SCHEMA_VERSION, "prediction_pool": C.PREDICTION_POOL,
          "selection_k": C.SELECTION_K, "export_k": C.EXPORT_K, "shortlist_size": C.SHORTLIST_SIZE,
          "final_display_size": C.FINAL_DISPLAY_SIZE, "score_semantics": C.SCORE_SEMANTICS,
          "pool_policy": ws.reports["pool_policy"], "pool_fingerprint": ws.reports["pool_fingerprint"],
          "eligibility_rules": C.ELIGIBILITY_RULES, "tie_order": "score descending, original article_id ascending"}
EXPORT_MANIFEST = {**common, "splits": contract, "all_training_and_export_requests": True,
                   "recorded_slates": RECORDED, "resource_estimate": BUDGET,
                   "labels": "joined separately from original recorded slates; test outcomes unavailable"}
METRICS = {"definitions": P.METRIC_DEFINITIONS, "selection": SELECTION, "selection_protocol": SELECTOR.protocol,
           "selection_validation": {**SELECTED_SAMPLE, "scope": SCOPES["selection_validation"]["info"]},
           "exported_validation": {**EXPORTED_VALIDATION, "scope": SCOPES["export_validation"]["info"]},
           "primary": "known_click_recall@100", "secondary": "observed_click_ndcg@100",
           "diagnostic_recorded_slates_validation": VAL_DIAGNOSTIC,
           "training_history": "training_history.json", "tuning": STUDY_SUMMARY,
           "test": "unavailable: official test has no click labels",
           "limitations": ["Validation reused for selection: development results.",
                           "Original ineligible/missed clicks remain in recall denominators and ideal DCG.",
                           "Observed-click recovery is exposure-biased; unshown outcomes remain unknown.",
                           "Training predictions are in-sample, not out-of-fold predictions."]}
RUN_CONFIG = {**common, "experiment_id": P.EXPERIMENT_ID, "run_id": RUN_ID, "owner": OWNER, "seed": SEED,
              "code": CODE_SOURCE, "environment": environment_info(), "config": CFG,
              "data_root": str(DATA_ROOT), "resolved_paths": ws.paths.as_dict(), "input_manifest": "input_manifest.json",
              "feature_order": FEATURE_ORDER, "feature_audit": F.FEATURE_AUDIT,
              "preprocessing": "model/preprocessing.json", "feature_dictionary": "model/feature_dictionary.json",
              "article_id_order": "model/article_ids.npy", "model_config": MODEL_CFG.to_json(), "parameters": PARAMS,
              "training_scope": TRAIN_SUBSET, "chosen_checkpoint": SELECTION["selected_checkpoint"],
              "selection_protocol": SELECTOR.protocol, "request_scopes": {key: value["info"] for key, value in SCOPES.items()},
              "request_scope_ids": "selection/request_scopes.json", "coverage": PREFLIGHT["coverage"],
              "recorded_ineligible": PREFLIGHT["recorded_ineligible"], "embedding_sources": ws.vector_reports,
              "history": ws.reports["history"], "export_counts": {split: result["summary"] for split, result in EXPORTS.items()},
              "tuning_objective": STUDY_SUMMARY["objective"], "training_objective": "BCE on supplied recorded pairs only",
              "persistent_location": str(PERSIST_DIR)}
for name, value in (("export_manifest.json", EXPORT_MANIFEST), ("metrics.json", METRICS), ("run_config.json", RUN_CONFIG)):
    atomic_write_json(RUN_DIR / name, value)
(RUN_DIR / "requirements.txt").write_text(P.requirements_text())
with log.stage("save complete run to shared Drive"):
    P.persist_run(RUN_DIR, PERSIST_DIR, log=log)
print("Persistent run:", PERSIST_DIR)


## 8. Reload check

Load the selected model, configuration and fitted preprocessing from the persistent Drive copy, without fitting. Compare a small recorded-slate and retained catalogue batch against the saved logits, with absolute tolerance `1e-4`. Check named feature order and original article-ID order too. The saved result is evidence only after this cell executes.

For later inference, add `<run>/model/code/` to the Python path, call `huim.persist.load_model(run, device)` and rebuild inputs with its returned preprocessing using `P.prepare_inputs(saved_config, data_root, pre=saved_preprocessor)`. Keep the saved feature/ID order and the same source snapshots.


In [ ]:
with log.stage("numerical reload from shared Drive"):
    RELOAD = P.reload_check(PERSIST_DIR, ws, CFG, DEVICE, n_requests=4, tolerance=1e-4)
atomic_write_json(RUN_DIR / "reload_check.json", RELOAD)
shutil.copyfile(RUN_DIR / "reload_check.json", PERSIST_DIR / "reload_check.json")
print(json.dumps(RELOAD, indent=2))
assert RELOAD["passed"], "Persistent reload did not reproduce the saved scores; inspect reload_check.json."


## 9. Business interpretation and handoff

Use the executed validation figures below to discuss whether candidate-aware history matching supports relevant shortlisting. Recall@100 remains the selection objective; the top-200 exports also support Recall@200 and later reranker prefix comparisons. Stage 2 receives the genuine shortlist and separate recorded-slate diagnostic scores for all three splits.

The complete persistent run includes tuned parameters, selected weights, fitted preprocessing, named features/category maps, original article-ID order, helper code, actual dependency versions, complete trial history/study snapshot, request scopes, metrics and numerical reload results. Later shared evaluation will use these artifacts for learning curves, search plots, representative recommendations and relevance/freshness/diversity/continuity comparisons. No model is retrained for that analysis.

Limitations remain: only exposed candidates have observed click outcomes; catalogue recovery is exposure-biased; metadata/vector snapshots do not prove historical production availability; offline clicks are a proxy, not measured retention or profit. Training predictions are in-sample. No test outcome analysis or reranker tuning belongs here.


In [ ]:
print("Selected validation sample:", {key: SELECTED_SAMPLE[key] for key in
                                      ("known_click_recall@100", "observed_click_ndcg@100")})
print("All validation requests:", {key: EXPORTED_VALIDATION[key] for key in
                                  ("known_click_recall@100", "observed_click_ndcg@100",
                                   "known_click_recall@200", "observed_click_ndcg@200")})
print("Actual tuning outcomes:", STUDY_SUMMARY["actual_trials"], STUDY_SUMMARY["states"])
print("Selected checkpoint:", SELECTION["selected_checkpoint"], "Stop reason:", TRAIN_STATE["stop_reason"])
print("Persistent reload passed:", RELOAD["passed"])
print("Handoff folder:", PERSIST_DIR)
